In [1]:
import os
import sys
import django
PATH_PROJECT = fr"D:\PROGRAMMING\Projects\siakad\siakad_django"
sys.path.append(PATH_PROJECT)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'siakad_django.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from django.conf import settings
from django.db.models import Q, F

In [2]:
from academic.models import (
    Kelas,
    MataKuliah,
    Dosen
)

In [3]:
from pathlib import Path
from io import BytesIO
import io

import typing

from openpyxl import load_workbook
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.utils import column_index_from_string
from openpyxl.utils.cell import coordinate_to_tuple

import pandas as pd

In [4]:
TEMPLATE_PATH = Path() / "template_jadwal.xlsx"
TEMPLATE_PATH.exists()

True

In [5]:
wb_template_jadwal = load_workbook(TEMPLATE_PATH, data_only=True)
ws_template_jadwal: Worksheet = wb_template_jadwal["template"]

meta_template_jadwal = pd.read_excel(TEMPLATE_PATH, sheet_name="meta")
meta_template_jadwal
value_dict = meta_template_jadwal.to_dict(orient="records")
len(value_dict)
value_dict

[{'no': 1,
  'field': 'C6',
  'label': 'slot_1_1_mtk',
  'type': 'text',
  'baris': 6,
  'kolom': 'C',
  'keterangan': 'senin',
  'slot': 1,
  'kelas_id': nan},
 {'no': 2,
  'field': 'C7',
  'label': 'slot_1_2_mtk',
  'type': 'text',
  'baris': 7,
  'kolom': 'C',
  'keterangan': 'senin',
  'slot': 2,
  'kelas_id': nan},
 {'no': 3,
  'field': 'C8',
  'label': 'slot_1_3_mtk',
  'type': 'text',
  'baris': 8,
  'kolom': 'C',
  'keterangan': 'senin',
  'slot': 3,
  'kelas_id': nan},
 {'no': 4,
  'field': 'C9',
  'label': 'slot_1_4_mtk',
  'type': 'text',
  'baris': 9,
  'kolom': 'C',
  'keterangan': 'senin',
  'slot': 4,
  'kelas_id': nan},
 {'no': 5,
  'field': 'C10',
  'label': 'slot_1_5_mtk',
  'type': 'text',
  'baris': 10,
  'kolom': 'C',
  'keterangan': 'senin',
  'slot': 5,
  'kelas_id': nan},
 {'no': 6,
  'field': 'C11',
  'label': 'slot_1_6_mtk',
  'type': 'text',
  'baris': 11,
  'kolom': 'C',
  'keterangan': 'senin',
  'slot': 6,
  'kelas_id': nan},
 {'no': 7,
  'field': 'C12',
 

In [6]:
# value_dict = {
#     row.get("field"): row
#     for row in meta_template_jadwal.to_dict(orient="records")
# }
# value_dict

data

In [7]:
class KelasSlot(typing.TypedDict):
    kelas_id: int
    field: str

In [8]:
kelas_slot_selected: list[KelasSlot] = [
    {"kelas_id": 130, "field": "C6"},
    {"kelas_id": 129, "field": "C7"},
    {"kelas_id": 131, "field": "C8"},
    {"kelas_id": 132, "field": "C9"},
    {"kelas_id": 133, "field": "C10"},
]

In [9]:
# kelas_selected = (
#     Kelas.objects
#     .filter(id__in=[slot["kelas_id"] for slot in selected_slot])
#     .select_related("mata_kuliah")
#     .values(
#         "id", "nama", "mata_kuliah__nama", "mata_kuliah__sks", "mata_kuliah__semester"
#     )
# )
# kelas_selected
# len(kelas_selected)

In [10]:
T = typing.TypeVar("T")

class SelectedField(typing.TypedDict, typing.Generic[T]):
    field: str
    kelas: T

In [11]:
from typing import TypedDict


class SelectedField(TypedDict):
    field: str


class CoordinateValue(TypedDict):
    coordinate: str
    fields: list[SelectedField]
    obj: object

In [12]:
class ExcelService:
    def __init__(self):
        pass

    @staticmethod
    def simpan_excel(wb) -> io.BytesIO:
        output = io.BytesIO()
        wb.save(output)
        output.seek(0)
        return output

    @staticmethod
    def sisipkan_array(
        ws, *, kolom: str, data: list, baris_awal: int = 1, coordinate=None
    ):
        if coordinate:
            baris, kolom = coordinate_to_tuple(coordinate=coordinate)

            for row, value in enumerate(data, start=baris):
                ws.cell(row=row, column=kolom, value=value)
        else:
            nomor_kolom = column_index_from_string(kolom)

            for row, value in enumerate(data, start=baris_awal):
                ws.cell(row=row, column=nomor_kolom, value=value)

    @staticmethod
    def sisipkan_nilai_by_coordinat(ws, *, coordinate: str, value: str):
        baris, kolom = coordinate_to_tuple(coordinate=coordinate)

        ws.cell(row=baris, column=kolom, value=value)
        
    @staticmethod
    def sisipkan_nilai_by_coordinate_dict_value(
        ws,
        *,
        values: list[CoordinateValue],
        obj_is_dict: bool = False
    ) -> None:
        for item in values:
            baris, kolom = coordinate_to_tuple(item["coordinate"])

            temp = (0, 3, 4, 5, 6, 10) # lokasi kolom
            for kolom_id, field in enumerate(item["fields"]):
                if obj_is_dict:
                    nilai = item["obj"].get(field["field"])
                else:
                    nilai = getattr(item["obj"], field["field"])
                
                kolom_real = temp[kolom_id] + kolom

                ws.cell(
                    row=baris,
                    column=kolom_real,
                    # column=kolom + kolom_id,
                    value=nilai,
                )
        

    # @staticmethod
    # def excecute(kelas_id: int, template_path: Path, **kwargs) -> Result:
    #     data = Query.mydata(kelas_id=kelas_id)
    #     daftar_mahasiswa = data.daftar_mahasiswa
    #     kelas = data.kelas
    #     krs = data.krs

    #     wb = load_workbook(template_path)
    #     ws_daftar_hadir = wb["KELAS"]
    #     ws_jurnal_perkuliahan = wb["JURNAL PERKULIAHAN"]

    #     dosen_nama = kwargs.get("DOSEN_PEMBINA", "")
    #     dosen_nidn = kwargs.get("NIDN", "")

    #     value_daftar_hadir: FieldDaftarHadirType = {
    #         "DOSEN_PEMBINA": dosen_nama,
    #         "NIDN": dosen_nidn,
    #         "DOSEN_PEMBINA_NIDN": dosen_nidn,
    #         "DOSEN_PEMBINA_NAMA": dosen_nama,
    #         "NAMA": [mahasiswa.nama for mahasiswa in daftar_mahasiswa],
    #         "NIM": [mahasiswa.nim for mahasiswa in daftar_mahasiswa],
    #         "WAKTU": kelas.waktu,
    #         "SEMESTER_RUANG": f"{angka_ke_romawi(kelas.mata_kuliah.semester)} / {kelas.ruangan or 'R207'}",
    #         "MATA_KULIAH_SKS": f"{kelas.mata_kuliah.nama} / {kelas.mata_kuliah.sks} SKS",
    #         "NAMA_KELAS": f"KELAS {kelas.nama.upper()}",
    #     }

    #     # 'GANJIL T.A. 2026/2027
        
    #     value_jurnal_perkuliahan: FieldJurnalType = {
    #         "DOSEN_PEMBINA": dosen_nama,
    #         "MATA_KULIAH_SKS": f"{kelas.mata_kuliah.nama.upper()} / {kelas.mata_kuliah.sks} SKS",
    #         "SEMESTER": angka_ke_romawi(kelas.mata_kuliah.semester),
    #         "TAHUN_AKADEMIK": f"{krs.semester.upper()} T.A {krs.tahun_ajaran}",
    #     }

    #     for key in fieldSet_daftar_hadir.keys():
    #         if key == "NAMA" or key == "NIM":
    #             continue
    #         ExcelService.sisipkan_nilai_by_coordinat(
    #             ws_daftar_hadir,
    #             coordinate=fieldSet_daftar_hadir[key],
    #             value=value_daftar_hadir[key],
    #         )
            
    #     for key in fieldSet_jurnal_perkuliahan.keys():
    #         ExcelService.sisipkan_nilai_by_coordinat(
    #             ws_jurnal_perkuliahan,
    #             coordinate=fieldSet_jurnal_perkuliahan[key],
    #             value=value_jurnal_perkuliahan[key],
    #         )

    #     # baris, kolom = coordinate_to_tuple(coordinate=fieldSet_daftar_hadir["NIM"])
    #     ExcelService.sisipkan_array(
    #         ws_daftar_hadir,
    #         kolom=fieldSet_daftar_hadir["NIM"],
    #         coordinate=fieldSet_daftar_hadir["NIM"],
    #         data=value_daftar_hadir["NIM"],
    #     )

    #     ExcelService.sisipkan_array(
    #         ws_daftar_hadir,
    #         kolom=fieldSet_daftar_hadir["NAMA"],
    #         coordinate=fieldSet_daftar_hadir["NAMA"],
    #         data=value_daftar_hadir["NAMA"],
    #     )

    #     # return ExcelService.simpan_excel(wb)
    #     return Result(is_success=True, data=ExcelService.simpan_excel(wb))

    #     # ExcelService.sisipkan_nilai_by_coordinat(ws, coordinate="E8", value=mata_kuliah.mata_kuliah)
    #     # ExcelService.sisipkan_nilai_by_coordinat(ws, coordinate="E9", value=mata_kuliah.semester)
    #     # ExcelService.sisipkan_nilai_by_coordinat(ws, coordinate="E10", value=mata_kuliah.dosen_pembina)
    #     # ExcelService.sisipkan_nilai_by_coordinat(ws, coordinate="E11", value=mata_kuliah.waktu)


In [13]:
# ExcelService.sisipkan_nilai_by_coordinate_dict_value(
#     ws,
#     coordinate="B5",
#     obj=mahasiswa,
#     fields=[
#         {"field": "nim"},
#         {"field": "nama"},
#         {"field": "kelas"},
#     ],
# )

In [14]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Kelas:
    mata_kuliah_nama: str
    mata_kuliah_sks: int
    mata_kuliah_smt: str
    kelas_nama: str
    kelas_ruangan: str
    dosen_nama: str

daftar_kelas = [
    Kelas(
        mata_kuliah_nama="Matematika",
        mata_kuliah_sks=3,
        kelas_nama="A",
        mata_kuliah_smt="I",
        kelas_ruangan="R207",
        dosen_nama="Budi"
    ),
    Kelas(
        mata_kuliah_nama="Bahasa Indonesia",
        mata_kuliah_sks=3,
        kelas_nama="B",
        mata_kuliah_smt="III",
        kelas_ruangan="R207",
        dosen_nama="Andi"
    ),
    Kelas(
        mata_kuliah_nama="Bahasa Inggris",
        mata_kuliah_sks=3,
        kelas_nama="C",
        mata_kuliah_smt="V",
        kelas_ruangan="R207",
        dosen_nama="Rudi"
    ),
]

In [15]:
kelas_slot_selected: list[KelasSlot] = [
    {"kelas_id": 130, "field": "C6"},
    {"kelas_id": 129, "field": "C7"},
    {"kelas_id": 131, "field": "C8"},
    {"kelas_id": 132, "field": "C9"},
    {"kelas_id": 133, "field": "C10"},
]

In [16]:
# ----- ganti nama -----
# key_mapping = {
#     "id": "kelas_id",
#     "nama": "kelas_nama",
#     "mata_kuliah__nama": "mata_kuliah",
#     "mata_kuliah__sks": "sks",
# }

# hasil = [
#     {
#         key_mapping.get(key, key): value
#         for key, value in item.items()
#     }
#     for item in data
# ]

In [17]:
obj_fields = [
    {"field": "mata_kuliah_nama"},
    {"field": "mata_kuliah_sks"},
    {"field": "mata_kuliah_smt"},
    {"field": "kelas_nama"},
    {"field": "dosen_nama"},
    {"field": "kelas_ruangan"},
]

SEMESTER = {
    1: "I",
    2: "II",
    3: "III",
    4: "IV",
    5: "V",
    6: "VI",
    7: "VII",
}

In [18]:
from academic.models import (
    Kelas,
    MataKuliah,
    Dosen
)

In [19]:
def kelas_slot_to_coordinate_value(*, kelas_slot_selected: list[KelasSlot]) -> list[CoordinateValue]:
    kelas_selected = (
        Kelas.objects
        .filter(id__in=[slot["kelas_id"] for slot in kelas_slot_selected])
        .select_related("mata_kuliah", "dosen")
        .values(
            "id", 
            kelas_nama=F("nama"), 
            kelas_ruangan=F("ruangan"), 
            dosen_nama=F("dosen__nama"), 
            mata_kuliah_nama= F("mata_kuliah__nama"), 
            mata_kuliah_sks=F("mata_kuliah__sks"), 
            mata_kuliah_smt=F("mata_kuliah__semester")
        )
    )
    
    kelas_by_id = {
        kelas["id"]: kelas
        for kelas in kelas_selected
    }
    
    
    hasil: list[CoordinateValue] = [
        {
            # "obj": kelas_by_id[slot["kelas_id"]],
            "coordinate": slot["field"],
            "obj": {
                **kelas_by_id[slot["kelas_id"]],
                "mata_kuliah_nama": kelas_by_id[slot["kelas_id"]]["mata_kuliah_nama"].upper(),
                "mata_kuliah_smt": SEMESTER.get(kelas_by_id[slot["kelas_id"]]["mata_kuliah_smt"], ""),
            },
            "fields": obj_fields
        }
        for slot in kelas_slot_selected
    ]
    
    return hasil

values = kelas_slot_to_coordinate_value(kelas_slot_selected=kelas_slot_selected)
values

[{'coordinate': 'C6',
  'obj': {'id': 130,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': None,
   'mata_kuliah_nama': 'ARTIFICIAL INTELLIGENCE (AI)',
   'mata_kuliah_sks': 3,
   'mata_kuliah_smt': 'III'},
  'fields': [{'field': 'mata_kuliah_nama'},
   {'field': 'mata_kuliah_sks'},
   {'field': 'mata_kuliah_smt'},
   {'field': 'kelas_nama'},
   {'field': 'dosen_nama'},
   {'field': 'kelas_ruangan'}]},
 {'coordinate': 'C7',
  'obj': {'id': 129,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': None,
   'mata_kuliah_nama': 'ARSITEKTUR DAN ORGANISASI KOMPUTER',
   'mata_kuliah_sks': 2,
   'mata_kuliah_smt': 'III'},
  'fields': [{'field': 'mata_kuliah_nama'},
   {'field': 'mata_kuliah_sks'},
   {'field': 'mata_kuliah_smt'},
   {'field': 'kelas_nama'},
   {'field': 'dosen_nama'},
   {'field': 'kelas_ruangan'}]},
 {'coordinate': 'C8',
  'obj': {'id': 131,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': None,
   'mata_kuliah_nama': 'B

values

In [20]:
# values_selected = {
#     "coordinates": ["C9", "C10"],
#     "kelas_ids": [130, 129]
# }


In [21]:
# kelas_dict = {
#     kelas.id: {
#         "id": kelas.id,
#         "nama": kelas.nama,
#         "mata_kuliah": {
#             "id": kelas.mata_kuliah.id,
#             "nama": kelas.mata_kuliah.nama,
#             "sks": kelas.mata_kuliah.sks,
#             "semester": kelas.mata_kuliah.semester,
#         },
#     }
#     for kelas in (
#         Kelas.objects
#         .filter(id__in=values_selected["kelas_ids"])
#         .select_related("mata_kuliah")
#     )
# }

# result = [
#     {
#         "coordinate": coordinate,
#         "kelas": kelas_dict[kelas_id],
#     }
#     for coordinate, kelas_id in zip(
#         values_selected["coordinates"],
#         values_selected["kelas_ids"],
#     )
#     if kelas_id in kelas_dict
# ]

# result

In [22]:
# kelas_selected = (
#     Kelas.objects
#     .filter(
#         id__in=values_selected["kelas_ids"]
#     )
#     .select_related("mata_kuliah")
#     .values("id", "mata_kuliah", "nama")
# )
# kelas_selected



In [23]:
# kelas_selected

In [24]:
# values = [
#     {
#         "coordinate": "C6",
#         "obj": daftar_kelas[0],
#         "fields": obj_fields,
#     },
#     {
#         "coordinate": "C7",
#         "obj": daftar_kelas[1],
#         "fields": obj_fields,
#     },
#     {
#         "coordinate": "C8",
#         "obj": daftar_kelas[2],
#         "fields": obj_fields,
#     },
# ]

# values = [
#     {
#         "coordinate": "C6",
#     }
#     for value in daftar_kelas
# ]

In [25]:
values

[{'coordinate': 'C6',
  'obj': {'id': 130,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': None,
   'mata_kuliah_nama': 'ARTIFICIAL INTELLIGENCE (AI)',
   'mata_kuliah_sks': 3,
   'mata_kuliah_smt': 'III'},
  'fields': [{'field': 'mata_kuliah_nama'},
   {'field': 'mata_kuliah_sks'},
   {'field': 'mata_kuliah_smt'},
   {'field': 'kelas_nama'},
   {'field': 'dosen_nama'},
   {'field': 'kelas_ruangan'}]},
 {'coordinate': 'C7',
  'obj': {'id': 129,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': None,
   'mata_kuliah_nama': 'ARSITEKTUR DAN ORGANISASI KOMPUTER',
   'mata_kuliah_sks': 2,
   'mata_kuliah_smt': 'III'},
  'fields': [{'field': 'mata_kuliah_nama'},
   {'field': 'mata_kuliah_sks'},
   {'field': 'mata_kuliah_smt'},
   {'field': 'kelas_nama'},
   {'field': 'dosen_nama'},
   {'field': 'kelas_ruangan'}]},
 {'coordinate': 'C8',
  'obj': {'id': 131,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': None,
   'mata_kuliah_nama': 'B

In [26]:
output = io.BytesIO()


In [27]:
# getattr(values, "obj")
# get(values, )

In [28]:
ExcelService.sisipkan_nilai_by_coordinate_dict_value(
    ws_template_jadwal,
    values=values,
    obj_is_dict=True
)

In [29]:
# for merged_range in ws_template_jadwal.merged_cells.ranges:
#     print(merged_range)

In [30]:
wb_template_jadwal.remove(wb_template_jadwal["meta"])
wb_template_jadwal.save("output1.xlsx")

In [31]:
# wb_template_jadwal.save(output)
# output.seek(0)

# with open("output.xlsx", "wb") as file:
#     file.write(output.getvalue())

In [32]:
# with open("output.xlsx", "wb") as file:
#     wb_template_jadwal.save(file)

In [33]:
values

[{'coordinate': 'C6',
  'obj': {'id': 130,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': None,
   'mata_kuliah_nama': 'ARTIFICIAL INTELLIGENCE (AI)',
   'mata_kuliah_sks': 3,
   'mata_kuliah_smt': 'III'},
  'fields': [{'field': 'mata_kuliah_nama'},
   {'field': 'mata_kuliah_sks'},
   {'field': 'mata_kuliah_smt'},
   {'field': 'kelas_nama'},
   {'field': 'dosen_nama'},
   {'field': 'kelas_ruangan'}]},
 {'coordinate': 'C7',
  'obj': {'id': 129,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': None,
   'mata_kuliah_nama': 'ARSITEKTUR DAN ORGANISASI KOMPUTER',
   'mata_kuliah_sks': 2,
   'mata_kuliah_smt': 'III'},
  'fields': [{'field': 'mata_kuliah_nama'},
   {'field': 'mata_kuliah_sks'},
   {'field': 'mata_kuliah_smt'},
   {'field': 'kelas_nama'},
   {'field': 'dosen_nama'},
   {'field': 'kelas_ruangan'}]},
 {'coordinate': 'C8',
  'obj': {'id': 131,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': None,
   'mata_kuliah_nama': 'B